In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/notebooks/faiazabrarzaman/arcadia-v2-phase-a/__results__.html
/kaggle/input/notebooks/faiazabrarzaman/arcadia-v2-phase-a/arcadia_v2_snp_matrix.pkl
/kaggle/input/notebooks/faiazabrarzaman/arcadia-v2-phase-a/__notebook__.ipynb
/kaggle/input/notebooks/faiazabrarzaman/arcadia-v2-phase-a/__output__.json
/kaggle/input/notebooks/faiazabrarzaman/arcadia-v2-phase-a/filtered_output.vcf.gz
/kaggle/input/notebooks/faiazabrarzaman/arcadia-v2-phase-a/custom.css
/kaggle/input/datasets/faiazabrarzaman/arcadia-presenceabsence-v2-ecoli-rawdownload/annotated_output.vcfgz
/kaggle/input/datasets/faiazabrarzaman/arcadia-presenceabsence-v2-ecoli-rawdownload/filtered_output.vcfgz
/kaggle/input/datasets/faiazabrarzaman/arcadia-presenceabsence-v2-ecoli-rawdownload/Ecoli_AMR_GenotypePhenotype/Ecoli_AMR_GenotypePhenotype/strains_metadata.csv
/kaggle/input/datasets/faiazabrarzaman/arcadia-presenceabsence-v2-ecoli-rawdownload/Ecoli_AMR_GenotypePhenotype/Ecoli_AMR_GenotypePhenotype/vcf_files/annotated_

In [2]:
import pickle
PKL_PATH = "/kaggle/input/notebooks/faiazabrarzaman/arcadia-v2-phase-a/arcadia_v2_snp_matrix.pkl"

with open(PKL_PATH, "rb") as f:
    snp_data = pickle.load(f)

print(f"Loaded shape: {snp_data['X_snp'].shape}")

Loaded shape: (6962, 535439)


In [3]:
"""
PHASE B STEP 1: BUILD FULL ARCADIA V2 SNP DATASET
"""

import pickle
import numpy as np
import pandas as pd
import re
from pathlib import Path
from sklearn.model_selection import train_test_split

# ============================================================================
# PATHS
# ============================================================================
SNP_PKL = "/kaggle/input/notebooks/faiazabrarzaman/arcadia-v2-phase-a/arcadia_v2_snp_matrix.pkl"
ARCADIA_BASE = Path("/kaggle/input/datasets/faiazabrarzaman/arcadia-presenceabsence-v2-ecoli-rawdownload/Ecoli_AMR_GenotypePhenotype/Ecoli_AMR_GenotypePhenotype")
PHENO_PATH = ARCADIA_BASE / "phenotype_matrix" / "phenotype_matrix_08302024.csv"
METADATA_PATH = ARCADIA_BASE / "strains_metadata.csv"
SRA_MAP_PATH = ARCADIA_BASE / "phenotype_matrix" / "SRA_to_genome_name.csv"

# ============================================================================
# LOAD SNP MATRIX
# ============================================================================
print("Loading SNP matrix...")
with open(SNP_PKL, "rb") as f:
    snp_data = pickle.load(f)

X_snp = snp_data["X_snp"]
positions = snp_data["snp_positions"]
genome_names_snp = snp_data["selected_genome_names"]
sra_ids_snp = snp_data["selected_sra_ids"]

print(f"  X_snp: {X_snp.shape}, dtype={X_snp.dtype}")
print(f"  Memory: {X_snp.nbytes / 1e9:.2f} GB")
print(f"  Genome names: {len(genome_names_snp)}")

# ============================================================================
# LOAD METADATA
# ============================================================================
print("\nLoading phenotype + metadata...")
pheno = pd.read_csv(PHENO_PATH)
metadata = pd.read_csv(METADATA_PATH)
sra_map = pd.read_csv(SRA_MAP_PATH)

print(f"  Phenotype: {pheno.shape}")
print(f"  Metadata: {metadata.shape}")
print(f"  SRA map: {sra_map.shape}")

# ============================================================================
# BUILD SAMPLE DATAFRAME (linking everything)
# ============================================================================
samples_df = pd.DataFrame({
    "snp_idx": np.arange(len(genome_names_snp)),
    "Genome.Name": genome_names_snp,
    "SRA.Accession": sra_ids_snp,
})

# Merge phenotype (preserve SNP order)
samples_df = samples_df.merge(pheno, on="Genome.Name", how="left")

# Merge metadata for phylogroup
samples_df = samples_df.merge(
    metadata[["Genome.Name", "MLST", "Other.Typing"]],
    on="Genome.Name", how="left"
)

print(f"\n  Final samples: {len(samples_df)}")
print(f"  Phenotype matched: {samples_df['ampicillin'].notna().sum()}")
print(f"  Metadata matched: {samples_df['MLST'].notna().sum()}")

Loading SNP matrix...


  X_snp: (6962, 535439), dtype=int8
  Memory: 3.73 GB
  Genome names: 6962

Loading phenotype + metadata...


  Phenotype: (6983, 52)
  Metadata: (6985, 88)
  SRA map: (6985, 3)

  Final samples: 6962
  Phenotype matched: 4868
  Metadata matched: 6821


In [4]:
"""
PHYLOGROUP + LABELS + SPLITS + SAVE
"""

# ============================================================================
# PHYLOGROUP DERIVATION
# ============================================================================
def parse_phylogroup(val):
    if pd.isna(val): return None
    m = re.findall(r'phylogroup:([A-Z][12]?)', str(val))
    return m[0] if m else None

def parse_mlst(val):
    if pd.isna(val): return None
    m = re.search(r'MLST\.Escherichia_coli_1\.(\d+)', str(val))
    return m.group(1) if m else None

ST_TO_PHYLOGROUP = {
    "131": "B2", "73": "B2", "95": "B2", "127": "B2", "12": "B2",
    "141": "B2", "144": "B2", "372": "B2", "404": "B2", "416": "B2",
    "428": "B2", "550": "B2", "1193": "B2", "14": "B2", "80": "B2",
    "498": "B2", "538": "B2", "983": "B2",
    "69": "D", "405": "D", "354": "D", "38": "D", "117": "D",
    "115": "D", "457": "D", "362": "D",
    "10": "A", "167": "A", "617": "A", "44": "A", "744": "A",
    "48": "A", "93": "A", "227": "A", "1284": "A", "168": "A",
    "752": "A", "224": "A",
    "11": "B1", "58": "B1", "155": "B1", "101": "B1", "162": "B1",
    "297": "B1", "446": "B1", "453": "B1", "517": "B1", "200": "B1",
    "62": "F", "59": "F", "522": "F", "648": "F",
    "88": "C", "410": "C", "23": "C",
    "350": "E", "57": "E",
}

samples_df["phylo_step1"] = samples_df["Other.Typing"].apply(parse_phylogroup)
samples_df["mlst_achtman"] = samples_df["MLST"].apply(parse_mlst)
samples_df["phylogroup_final"] = (
    samples_df["phylo_step1"]
    .fillna(samples_df["mlst_achtman"].map(ST_TO_PHYLOGROUP))
    .fillna("Unknown")
)

print("Phylogroup distribution:")
print(samples_df["phylogroup_final"].value_counts())

unique_phylogroups = sorted(samples_df["phylogroup_final"].unique())
phylogroup_to_int = {pg: i for i, pg in enumerate(unique_phylogroups)}
cluster_labels = np.array([phylogroup_to_int[pg] for pg in samples_df["phylogroup_final"]], dtype=np.int32)

# ============================================================================
# LABELS
# ============================================================================
THESIS_AB = ["AMP", "CIP", "AMC", "SXT", "GEN", "CTX", "CAZ"]
AB_COL_MAP = {
    "AMP": "ampicillin", "CIP": "ciprofloxacin",
    "AMC": "amoxicillin/clavulanic acid", "SXT": "trimethoprim/sulfamethoxazole",
    "GEN": "gentamicin", "CTX": "cefotaxime", "CAZ": "ceftazidime",
}

def pheno_to_binary(val):
    if pd.isna(val): return np.nan
    s = str(val).strip().lower()
    if s in {"r", "res", "resistant", "i", "int", "intermediate"}: return 1.0
    if s in {"s", "sus", "susceptible"}: return 0.0
    return np.nan

n_samples = len(samples_df)
Y = np.full((n_samples, len(THESIS_AB)), np.nan, dtype=np.float32)
for i, ab in enumerate(THESIS_AB):
    Y[:, i] = samples_df[AB_COL_MAP[ab]].apply(pheno_to_binary).values

label_mask = (~np.isnan(Y)).astype(bool)
Y_filled = np.nan_to_num(Y, nan=0.0).astype(bool)

print("\nLabel distribution:")
for i, ab in enumerate(THESIS_AB):
    valid = label_mask[:, i]
    pos = (Y[valid, i] == 1).sum()
    neg = (Y[valid, i] == 0).sum()
    print(f"  {ab}: pos={pos}, neg={neg}, missing={(~valid).sum()}, prev={pos/(pos+neg):.3f}")

# ============================================================================
# RANDOM SPLIT (stratified on AMP)
# ============================================================================
amp_idx = THESIS_AB.index("AMP")
strat = np.where(np.isnan(Y[:, amp_idx]), -1, Y[:, amp_idx].astype(int))

idx_all = np.arange(n_samples)
idx_trainval, idx_test = train_test_split(idx_all, test_size=0.15, random_state=42, stratify=strat)
idx_train, idx_val = train_test_split(idx_trainval, test_size=0.15/0.85, random_state=42, stratify=strat[idx_trainval])

random_split = {"train_idx": idx_train, "val_idx": idx_val, "test_idx": idx_test}
print(f"\nRandom split: {len(idx_train)}/{len(idx_val)}/{len(idx_test)}")

# ============================================================================
# PHYLOGENY SPLIT (stratified on phylogroup)
# ============================================================================
idx_trainval_p, idx_test_p = train_test_split(idx_all, test_size=0.15, random_state=42, stratify=cluster_labels)
idx_train_p, idx_val_p = train_test_split(
    idx_trainval_p, test_size=0.15/0.85, random_state=42,
    stratify=cluster_labels[idx_trainval_p]
)

phylogeny_split = {"train_idx": idx_train_p, "val_idx": idx_val_p, "test_idx": idx_test_p}
print(f"Phylogeny split: {len(idx_train_p)}/{len(idx_val_p)}/{len(idx_test_p)}")

# ============================================================================
# SAVE FULL DATASET
# ============================================================================
full_dataset = {
    "X": X_snp,  # int8 (samples, 535439)
    "Y": Y_filled,
    "label_mask": label_mask,
    "genome_names": np.array(samples_df["Genome.Name"].values),
    "sra_ids": np.array(samples_df["SRA.Accession"].values),
    "snp_positions": positions,
    "label_names": np.array(THESIS_AB),
    "antibiotic_to_index": {ab: i for i, ab in enumerate(THESIS_AB)},
    "cluster_labels": cluster_labels,
    "phylogroup_names": np.array(unique_phylogroups),
    "phylogroup_to_int": phylogroup_to_int,
    "random_split": random_split,
    "phylogeny_split": phylogeny_split,
    "n_samples": n_samples,
    "n_features": X_snp.shape[1],
    "n_antibiotics": len(THESIS_AB),
    "n_clusters": len(unique_phylogroups),
    "preprocess_meta": {
        "source": "Arcadia v2 VCF (Zenodo 14364732)",
        "feature_type": "SNP integer encoding (0-3)",
        "n_snps_kept": X_snp.shape[1],
        "n_streamed_total": snp_data["n_streamed"],
        "label_encoding": "R+I=1, S=0, missing=NaN",
        "phylogroup_source": "Other.Typing + MLST→phylogroup lookup",
        "version": "v2_phase_b_vcf",
    },
}

OUT_PATH = Path("/kaggle/working/arcadia_v2_full_dataset.pkl")
with open(OUT_PATH, "wb") as f:
    pickle.dump(full_dataset, f, protocol=4)

print(f"\n✅ Saved: {OUT_PATH}")
print(f"   Size: {OUT_PATH.stat().st_size / 1e9:.2f} GB")

Phylogroup distribution:
phylogroup_final
B2         3542
Unknown    1697
D           723
A           317
F           307
B1          258
C           114
E             4
Name: count, dtype: int64

Label distribution:
  AMP: pos=2439, neg=2429, missing=2094, prev=0.501
  CIP: pos=1377, neg=4656, missing=929, prev=0.228
  AMC: pos=1180, neg=2598, missing=3184, prev=0.312
  SXT: pos=1060, neg=2358, missing=3544, prev=0.310
  GEN: pos=710, neg=5327, missing=925, prev=0.118
  CTX: pos=823, neg=4586, missing=1553, prev=0.152
  CAZ: pos=780, neg=5193, missing=989, prev=0.131

Random split: 4872/1045/1045
Phylogeny split: 4872/1045/1045


/tmp/ipykernel_73/1954517407.py:86: RuntimeWarning: invalid value encountered in cast
  strat = np.where(np.isnan(Y[:, amp_idx]), -1, Y[:, amp_idx].astype(int))



✅ Saved: /kaggle/working/arcadia_v2_full_dataset.pkl
   Size: 3.95 GB


In [5]:
"""
PHASE B STEP 2.1: XGBoost on PCA-256 of SNP features
"""
!pip install xgboost
import pickle
import numpy as np
import xgboost as xgb
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import average_precision_score, matthews_corrcoef
import time

# Load
with open("/kaggle/working/arcadia_v2_full_dataset.pkl", "rb") as f:
    ds = pickle.load(f)

X = ds["X"].astype(np.float32)  # convert int8 → float32 for PCA
Y = ds["Y"]
mask = ds["label_mask"]
labels = ds["label_names"]
rs = ds["random_split"]
tr, va, te = rs["train_idx"], rs["val_idx"], rs["test_idx"]

print(f"X shape: {X.shape}")
print(f"Train/Val/Test: {len(tr)}/{len(va)}/{len(te)}")

# PCA-256
print("\nPCA dim search...")
pca_results = {}
for dim in [128, 256, 512, 1024, 2048]:
    print(f"\n  PCA dim={dim}")
    t0 = time.time()
    pca = TruncatedSVD(n_components=dim, random_state=42)
    Xtr_pca = pca.fit_transform(X[tr])
    Xva_pca = pca.transform(X[va])
    Xte_pca = pca.transform(X[te])
    var = pca.explained_variance_ratio_.sum()
    print(f"    Time: {time.time()-t0:.1f}s, Var: {var:.3f}")
    
    pr_aucs = []
    mccs = []
    for i, ab in enumerate(labels):
        m_tr = mask[tr, i]
        m_va = mask[va, i]
        m_te = mask[te, i]
        if m_tr.sum() < 50:
            pr_aucs.append(np.nan); mccs.append(np.nan); continue
        
        clf = xgb.XGBClassifier(
            n_estimators=300, max_depth=5, learning_rate=0.08,
            random_state=42, n_jobs=-1, verbosity=0, tree_method="hist"
        )
        clf.fit(Xtr_pca[m_tr], Y[tr, i][m_tr])
        
        pred_te = clf.predict_proba(Xte_pca[m_te])[:, 1]
        pr = average_precision_score(Y[te, i][m_te], pred_te)
        pred_te_bin = (pred_te > 0.5).astype(int)
        mcc = matthews_corrcoef(Y[te, i][m_te], pred_te_bin)
        
        pr_aucs.append(pr)
        mccs.append(mcc)
        print(f"    {ab}: PR={pr:.4f} MCC={mcc:.4f}")
    
    avg_pr = np.nanmean(pr_aucs)
    avg_mcc = np.nanmean(mccs)
    print(f"    AVG PR={avg_pr:.4f} MCC={avg_mcc:.4f}")
    pca_results[dim] = {"avg_pr": avg_pr, "avg_mcc": avg_mcc, "per_ab_pr": dict(zip(labels, pr_aucs))}

print("\n" + "="*60)
print("PCA SEARCH RESULTS (TEST set)")
print("="*60)
for dim, r in pca_results.items():
    print(f"  dim={dim}: PR={r['avg_pr']:.4f}, MCC={r['avg_mcc']:.4f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/131.7 MB ? eta -:--:--

   ━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.3/131.7 MB 201.7 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━ 81.0/131.7 MB 222.3 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━ 114.0/131.7 MB 199.2 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 131.6/131.7 MB 193.3 MB/s eta 0:00:01

...

    AMC: PR=0.7024 MCC=0.4616


    SXT: PR=0.7226 MCC=0.4889


    GEN: PR=0.6357 MCC=0.4887


    CTX: PR=0.8630 MCC=0.7355


    CAZ: PR=0.7803 MCC=0.6447
    AVG PR=0.7895 MCC=0.6103

PCA SEARCH RESULTS (TEST set)
  dim=128: PR=0.7815, MCC=0.6199
  dim=256: PR=0.7918, MCC=0.6320
  dim=512: PR=0.7924, MCC=0.6285
  dim=1024: PR=0.7992, MCC=0.6160
  dim=2048: PR=0.7895, MCC=0.6103


In [6]:
"""
BUILD MASTER FEATURE DATASET
"""

import pickle, numpy as np, pandas as pd, re, os, gc
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

# ============================================================================
# PATHS
# ============================================================================
SNP_PKL = "/kaggle/input/notebooks/faiazabrarzaman/arcadia-v2-phase-a/arcadia_v2_snp_matrix.pkl"
ARCADIA_BASE = Path("/kaggle/input/datasets/faiazabrarzaman/arcadia-presenceabsence-v2-ecoli-rawdownload/Ecoli_AMR_GenotypePhenotype/Ecoli_AMR_GenotypePhenotype")

# ============================================================================
# LOAD SNP MATRIX
# ============================================================================
print("Loading SNP matrix (4 GB)...")
with open(SNP_PKL, "rb") as f:
    snp_data = pickle.load(f)

X_snp = snp_data["X_snp"]
genome_names = snp_data["selected_genome_names"]
sra_ids = snp_data["selected_sra_ids"]
print(f"  X_snp: {X_snp.shape}")

# ============================================================================
# BUILD METADATA
# ============================================================================
print("\nBuilding metadata...")
pheno = pd.read_csv(ARCADIA_BASE / "phenotype_matrix" / "phenotype_matrix_08302024.csv")
metadata = pd.read_csv(ARCADIA_BASE / "strains_metadata.csv")

samples_df = pd.DataFrame({
    "Genome.Name": genome_names,
    "SRA.Accession": sra_ids,
})
samples_df = samples_df.merge(pheno, on="Genome.Name", how="left")
samples_df = samples_df.merge(metadata[["Genome.Name", "MLST", "Other.Typing"]], on="Genome.Name", how="left")

# Phylogroup
def parse_phylo(v):
    if pd.isna(v): return None
    m = re.findall(r'phylogroup:([A-Z][12]?)', str(v))
    return m[0] if m else None

def parse_mlst(v):
    if pd.isna(v): return None
    m = re.search(r'MLST\.Escherichia_coli_1\.(\d+)', str(v))
    return m.group(1) if m else None

ST_MAP = {"131":"B2","73":"B2","95":"B2","127":"B2","12":"B2","141":"B2","144":"B2","372":"B2","404":"B2","416":"B2","428":"B2","550":"B2","1193":"B2","14":"B2","80":"B2","498":"B2","538":"B2","983":"B2","69":"D","405":"D","354":"D","38":"D","117":"D","115":"D","457":"D","362":"D","10":"A","167":"A","617":"A","44":"A","744":"A","48":"A","93":"A","227":"A","1284":"A","168":"A","752":"A","224":"A","11":"B1","58":"B1","155":"B1","101":"B1","162":"B1","297":"B1","446":"B1","453":"B1","517":"B1","200":"B1","62":"F","59":"F","522":"F","648":"F","88":"C","410":"C","23":"C","350":"E","57":"E"}

samples_df["phylogroup_final"] = (
    samples_df["Other.Typing"].apply(parse_phylo)
    .fillna(samples_df["MLST"].apply(parse_mlst).map(ST_MAP))
    .fillna("Unknown")
)

unique_pg = sorted(samples_df["phylogroup_final"].unique())
pg_to_int = {pg: i for i, pg in enumerate(unique_pg)}
cluster_labels = np.array([pg_to_int[pg] for pg in samples_df["phylogroup_final"]], dtype=np.int32)

# Phylogroup one-hot encoding
phylogroup_onehot = np.zeros((len(samples_df), len(unique_pg)), dtype=np.float32)
phylogroup_onehot[np.arange(len(samples_df)), cluster_labels] = 1.0

print(f"  Phylogroups: {unique_pg}")
print(f"  Cluster distribution: {dict(zip(unique_pg, np.bincount(cluster_labels)))}")

# Labels
THESIS_AB = ["AMP","CIP","AMC","SXT","GEN","CTX","CAZ"]
AB_MAP = {"AMP":"ampicillin","CIP":"ciprofloxacin","AMC":"amoxicillin/clavulanic acid","SXT":"trimethoprim/sulfamethoxazole","GEN":"gentamicin","CTX":"cefotaxime","CAZ":"ceftazidime"}

def to_bin(v):
    if pd.isna(v): return np.nan
    s = str(v).strip().lower()
    if s in {"r","resistant","i","intermediate"}: return 1.0
    if s in {"s","susceptible"}: return 0.0
    return np.nan

n = len(samples_df)
Y = np.full((n, 7), np.nan, dtype=np.float32)
for i, ab in enumerate(THESIS_AB):
    Y[:, i] = samples_df[AB_MAP[ab]].apply(to_bin).values
label_mask = (~np.isnan(Y)).astype(bool)
Y_bool = np.nan_to_num(Y, nan=0.0).astype(bool)

# Splits
amp_strat = np.where(np.isnan(Y[:,0]), -1, Y[:,0].astype(int))
idx_all = np.arange(n)
idx_tv, idx_te = train_test_split(idx_all, test_size=0.15, random_state=42, stratify=amp_strat)
idx_tr, idx_va = train_test_split(idx_tv, test_size=0.15/0.85, random_state=42, stratify=amp_strat[idx_tv])

idx_tv_p, idx_te_p = train_test_split(idx_all, test_size=0.15, random_state=42, stratify=cluster_labels)
idx_tr_p, idx_va_p = train_test_split(idx_tv_p, test_size=0.15/0.85, random_state=42, stratify=cluster_labels[idx_tv_p])

print(f"  Random split: {len(idx_tr)}/{len(idx_va)}/{len(idx_te)}")
print(f"  Phylogeny split: {len(idx_tr_p)}/{len(idx_va_p)}/{len(idx_te_p)}")

# ============================================================================
# COMPUTE PCA FEATURES (heavy step)
# ============================================================================
print("\nConverting X to float32 for PCA...")
X_float = X_snp.astype(np.float32)
del X_snp; gc.collect()

print("Fitting PCA-1024 on train...")
pca = TruncatedSVD(n_components=1024, random_state=42)
Xtr_pca = pca.fit_transform(X_float[idx_tr])
Xva_pca = pca.transform(X_float[idx_va])
Xte_pca = pca.transform(X_float[idx_te])
var_retained = pca.explained_variance_ratio_.sum()
print(f"  Variance retained: {var_retained:.3f}")

# Free X memory
del X_float; gc.collect()

# Standardize
print("Standardizing...")
scaler = StandardScaler()
Xtr_pca = scaler.fit_transform(Xtr_pca).astype(np.float32)
Xva_pca = scaler.transform(Xva_pca).astype(np.float32)
Xte_pca = scaler.transform(Xte_pca).astype(np.float32)
print(f"  PCA features: {Xtr_pca.shape}")

# ============================================================================
# BUILD MASTER PKL
# ============================================================================
master = {
    # PCA features (Stage 1)
    "X_pca_train": Xtr_pca,
    "X_pca_val": Xva_pca,
    "X_pca_test": Xte_pca,
    "pca_var_retained": var_retained,
    "pca_n_components": 1024,
    
    # Phylogeny features (Stage 3)
    "cluster_labels": cluster_labels,
    "phylogroup_onehot": phylogroup_onehot,
    "phylogroup_names": np.array(unique_pg),
    "phylogroup_to_int": pg_to_int,
    
    # Labels & masks
    "Y": Y_bool,
    "label_mask": label_mask,
    "label_names": np.array(THESIS_AB),
    "antibiotic_to_index": {ab: i for i, ab in enumerate(THESIS_AB)},
    
    # Splits
    "random_split": {"train_idx": idx_tr, "val_idx": idx_va, "test_idx": idx_te},
    "phylogeny_split": {"train_idx": idx_tr_p, "val_idx": idx_va_p, "test_idx": idx_te_p},
    
    # Sample IDs
    "genome_names": np.array(genome_names),
    "sra_ids": np.array(sra_ids),
    
    # Metadata
    "n_samples": n,
    "n_pca_features": Xtr_pca.shape[1],
    "n_antibiotics": 7,
    "n_phylogroups": len(unique_pg),
    "preprocess_meta": {
        "source": "Arcadia v2 VCF (Zenodo 14364732)",
        "feature_type": "SNP integer encoding (0-3) → PCA-1024",
        "n_snps_original": 535439,
        "label_encoding": "R+I=1, S=0, missing=NaN",
        "phylogroup_source": "Other.Typing + MLST→phylogroup lookup",
        "version": "v2_master_features",
    },
}

# ============================================================================
# SAVE
# ============================================================================
OUT = Path("/kaggle/working/arcadia_v2_master_features.pkl")
with open(OUT, "wb") as f:
    pickle.dump(master, f, protocol=4)

size_mb = OUT.stat().st_size / 1e6
print(f"\n✅ MASTER DATASET SAVED")
print(f"  Path: {OUT}")
print(f"  Size: {size_mb:.2f} MB")
print(f"  Contents:")
print(f"    PCA train/val/test: {Xtr_pca.shape}/{Xva_pca.shape}/{Xte_pca.shape}")
print(f"    Phylogroup one-hot: {phylogroup_onehot.shape}")
print(f"    Y: {Y_bool.shape}, mask: {label_mask.shape}")
print(f"    Random split: {len(idx_tr)}/{len(idx_va)}/{len(idx_te)}")
print(f"    Phylogeny split: {len(idx_tr_p)}/{len(idx_va_p)}/{len(idx_te_p)}")

Loading SNP matrix (4 GB)...


  X_snp: (6962, 535439)

Building metadata...


  Phylogroups: ['A', 'B1', 'B2', 'C', 'D', 'E', 'F', 'Unknown']
  Cluster distribution: {'A': np.int64(317), 'B1': np.int64(258), 'B2': np.int64(3542), 'C': np.int64(114), 'D': np.int64(723), 'E': np.int64(4), 'F': np.int64(307), 'Unknown': np.int64(1697)}
  Random split: 4872/1045/1045
  Phylogeny split: 4872/1045/1045

Converting X to float32 for PCA...


/tmp/ipykernel_73/1909695719.py:92: RuntimeWarning: invalid value encountered in cast
  amp_strat = np.where(np.isnan(Y[:,0]), -1, Y[:,0].astype(int))


Fitting PCA-1024 on train...


  Variance retained: 0.947
Standardizing...
  PCA features: (4872, 1024)

✅ MASTER DATASET SAVED
  Path: /kaggle/working/arcadia_v2_master_features.pkl
  Size: 30.76 MB
  Contents:
    PCA train/val/test: (4872, 1024)/(1045, 1024)/(1045, 1024)
    Phylogroup one-hot: (6962, 8)
    Y: (6962, 7), mask: (6962, 7)
    Random split: 4872/1045/1045
    Phylogeny split: 4872/1045/1045
